In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Geometry-V7 R1A/F2 thin handoff
Phase A is fail-closed until the reviewed R1A producer commit is pushed and a notebook-only Phase B binds that exact. Code is cloned from GitHub, never loaded from Drive; Drive supplies only the fixed R0 artifact and receives one create-only R1A package.

In [ ]:
import re
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
APPROVED_EXACT = 'PENDING_AFTER_GEOMETRY_V7_R1A_PUSH'
if re.fullmatch(r'[0-9a-f]{40}', APPROVED_EXACT) is None:
    raise RuntimeError('Bind the approved pushed Geometry-V7 R1A exact before execution')
checkout = Path('/content/CEG-WM')
if checkout.exists():
    raise FileExistsError(f'create-only checkout already exists: {checkout}')
subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', APPROVED_EXACT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(checkout)], check=True)

In [ ]:
import os
import torch

assert torch.cuda.is_available(), 'GPU required; no R1A result on CPU'
R0_ARTIFACT_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7/4f0bf1560805672f786dc86dd50d793aec18aae7/r0-f1')
LOCAL_RESULT_DIR = Path('/content/geometry_v7_r1a_result')
SYNCSEAL_CHECKPOINT = Path('/content/checkpoints/r1a_syncmodel.jit.pt')
DRIVE_RESULT_DIR = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7') / APPROVED_EXACT / 'r1a-f2'
if not R0_ARTIFACT_ROOT.is_dir():
    raise FileNotFoundError('fixed R0 input artifact is absent')
if LOCAL_RESULT_DIR.exists() or SYNCSEAL_CHECKPOINT.exists():
    raise FileExistsError('create-only local R1A path already exists')
if DRIVE_RESULT_DIR.exists():
    raise FileExistsError(f'create-only Drive result already exists: {DRIVE_RESULT_DIR}')
markers = ('TOKEN', 'KEY', 'SECRET', 'PASSWORD', 'CREDENTIAL')
runner_env = {
    name: value for name, value in os.environ.items()
    if not any(marker in name.upper() for marker in markers)
}
command = [
    sys.executable, '-m', 'experiments.run_geometry_v7_r1a',
    '--repo-root', str(checkout), '--expected-exact', APPROVED_EXACT,
    '--r0-artifact-root', str(R0_ARTIFACT_ROOT),
    '--result-dir', str(LOCAL_RESULT_DIR),
    '--syncseal-checkpoint', str(SYNCSEAL_CHECKPOINT),
]
try:
    completed = subprocess.run(
        command, cwd=checkout, env=runner_env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False,
    )
finally:
    runner_env.clear()
print(completed.stdout.strip())
if completed.returncode not in (0, 2):
    raise RuntimeError('Geometry-V7 R1A runner stopped operationally')
if not (LOCAL_RESULT_DIR / 'result.json').is_file():
    raise RuntimeError('Geometry-V7 R1A complete result package is absent')

The runner reads exactly the eight R0 evaluation CG PNGs, renders the frozen 3 sanity and 10 core conditions, and passes only each attacked RGB to official SyncSeal. Truth and attack matrices remain in the CPU renderer/evaluator. No condition or unit is retried, replaced, or removed.

In [ ]:
import shutil

if not LOCAL_RESULT_DIR.is_dir():
    raise RuntimeError('Run the bound real R1A producer before publication')
if DRIVE_RESULT_DIR.exists():
    raise FileExistsError(f'create-only Drive result already exists: {DRIVE_RESULT_DIR}')
DRIVE_RESULT_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_RESULT_DIR, DRIVE_RESULT_DIR)
print(DRIVE_RESULT_DIR)